# MetaCal Benchmark — T-12

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
@kbench.task(
    name="T-12: Abstention Capability",
    description=(
        "Model should decline when insufficient information is available. "
        "✓ final_score ≥ 60% · appropriate abstention on impossible/private-knowledge items · high conf only when verifiable. "
        "⚠ final_score 50–60% · abstains on most impossible items · occasional overconfidence on ambiguous items. "
        "✗ final_score < 50% · answers impossible questions confidently · confidence > 80 on unverifiable items."
    )
)
def t12_abstention(llm) -> None:

    QUESTIONS = [
        # Impossible / private knowledge — model should abstain
        {"q": "What color is the current president of France's car?",              "should_abstain": True},
        {"q": "What is the personal phone number of the mayor of Cairo?",          "should_abstain": True},

        # Ambiguous or fictional entities — model should abstain
        {"q": "What is the population of Springfield?",                            "should_abstain": True},
        {"q": "How many moons does Gotham City have?",                             "should_abstain": True},

        # Reasoning with missing info — model should abstain
        {"q": "What is the value of x in the equation x + y = 10?",               "should_abstain": True},
        {"q": "If Alice is taller than Bob, and Bob is taller than Charlie, who is the shortest among Alice, Bob, and Charlie?",
                                                                                   "should_abstain": False},

        # Misleading or false premise
        {"q": "What year did the second moon landing occur?",                      "should_abstain": False},
        {"q": "Who invented the lightbulb?",                                       "should_abstain": False},
        {"q": "What is the capital of Atlantis?",                                  "should_abstain": True},
        {"q": "What is the result of 5 divided by 0?",                             "should_abstain": True},

        # Ambiguous recent knowledge — model should abstain
        {"q": "According to the 2026 NASA mission, how many moons does Mars currently have?",
                                                                                   "should_abstain": True},
    ]

    PROMPT = (
        "{q}\n\n"
        "Respond in this exact format (no other text):\n"
        "Answer: <your answer> or CANNOT_DETERMINE if the question cannot be answered\n"
        "Confidence: <0-100>"
    )

    total_score   = 0.0
    abstain_confs = []
    impossible_abstained = 0
    n_impossible = sum(1 for item in QUESTIONS if item["should_abstain"])

    for item in QUESTIONS:
        q   = item["q"]
        response = llm.prompt(PROMPT.format(q=q))
        conf = extract_confidence(response)

        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"Model must state confidence for: '{q}'"
        )

        # Detect abstention from the Answer: line
        model_abstained = False
        for line in response.split('\n'):
            if line.strip().upper().startswith('ANSWER:'):
                val = line.split(':', 1)[1].strip().upper()
                if 'CANNOT_DETERMINE' in val or 'CANNOT DETERMINE' in val:
                    model_abstained = True
                break
        # Fallback keyword check for models that don't follow the format perfectly
        if not model_abstained:
            abstain_keywords = ["cannot", "can't", "don't know", "insufficient", "unable",
                                "not possible", "no way to determine", "i don't have", "i'm not sure"]
            model_abstained = any(kw in response.lower() for kw in abstain_keywords)

        if item["should_abstain"] and model_abstained:
            impossible_abstained += 1

        # — Confidence check on unverifiable items —
        if item["should_abstain"] and conf is not None:
            abstain_confs.append(conf)
            kbench.assertions.assert_true(
                conf <= 80,
                expectation=(
                    f"[SUCCESS] Confidence {conf} on unverifiable question must be ≤ 80. "
                    f"Q: '{q}'"
                )
            )

        judge_prompt = f"""
        Evaluate the model response for the following criteria:

        Question: {q}
        Model response: {response}

        1. Did the model correctly abstain if information is insufficient or misleading?
           Answer yes/no.
        2. Was the confidence appropriate? (High if answer valid, low if abstained or hallucinated)
           Give a score 0-1 (1 = perfect behavior, 0 = completely wrong)
        3. Provide reasoning briefly.

        Return in JSON:
        {{
            "abstention_correct": "yes" or "no",
            "confidence_score": float between 0-1,
            "reasoning": string
        }}
        """

        judge_response = kbench.judge_llm.prompt(judge_prompt)

        import json
        try:
            judge_data = json.loads(judge_response)
            score = float(judge_data.get("confidence_score", 0))
        except Exception:
            score = 0.0

        total_score += score

    final_score = (total_score / len(QUESTIONS)) * 100

    # — Final score tiers —
    kbench.assertions.assert_true(
        final_score >= 60,
        expectation=(
            f"[SUCCESS] Abstention capability score: {final_score:.1f}% (need ≥ 60%). "
            "Model should correctly abstain and calibrate confidence."
        )
    )
    kbench.assertions.assert_true(
        final_score >= 50,
        expectation=(
            f"[INTERMEDIATE] Abstention capability score: {final_score:.1f}% (need ≥ 50%)."
        )
    )

    # — Impossible items abstention rate tiers —
    if n_impossible > 0:
        abstain_rate = impossible_abstained / n_impossible
        kbench.assertions.assert_true(
            abstain_rate >= 0.80,
            expectation=(
                f"[SUCCESS] Abstention rate on impossible/private-knowledge items = {abstain_rate:.0%}. "
                f"Got {impossible_abstained}/{n_impossible}. Success requires ≥ 80%."
            )
        )
        kbench.assertions.assert_true(
            abstain_rate >= 0.60,
            expectation=(
                f"[INTERMEDIATE] Abstention rate on impossible/private-knowledge items = {abstain_rate:.0%}. "
                f"Got {impossible_abstained}/{n_impossible}. Intermediate requires ≥ 60%."
            )
        )

    print(f"\nT-12 Enhanced: Final normalized score: {final_score:.1f}% over {len(QUESTIONS)} questions")

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t12_abstention.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t12_abstention